In [ ]:
# ============================================================
# 03_final_feature_engineering
# Thesis final pipeline
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

print("OUTPUT_02:", OUTPUT_02)
print("OUTPUT_03:", OUTPUT_03)

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

df_cohort = pd.read_parquet(OUTPUT_02 / "02_final_cohort.parquet")
df_labs = pd.read_parquet(OUTPUT_01 / "01_labs_selected.parquet")

print("Final cohort:", df_cohort.shape)
print("Labs selected:", df_labs.shape)

print("Cohort episodes:", df_cohort["episode_key"].nunique())
print("Lab episodes:", df_labs["episode_key"].nunique())

In [ ]:
# ============================================================
# STANDARDIZE KEYS AND DATES
# ============================================================

for df in [df_cohort, df_labs]:
    df["episode_key"] = (
        df["episode_key"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

df_cohort["startDate"] = pd.to_datetime(df_cohort["startDate"], errors="coerce")
df_cohort["endDate"] = pd.to_datetime(df_cohort["endDate"], errors="coerce")

df_labs["Date"] = pd.to_datetime(df_labs["Date"], errors="coerce")
df_labs["Numeric_value"] = pd.to_numeric(df_labs["Numeric_value"], errors="coerce")

print("Date and key standardization completed.")

In [ ]:
# ============================================================
# KEEP LABS FROM FINAL COHORT ONLY
# ============================================================

cohort_episode_dates = df_cohort[
    ["episode_key", "startDate", "endDate"]
].copy()

df_labs_cohort = df_labs.merge(
    cohort_episode_dates,
    on="episode_key",
    how="inner",
    suffixes=("", "_cohort")
)

print("Labs assigned to final cohort:", df_labs_cohort.shape)
print("Episodes with labs:", df_labs_cohort["episode_key"].nunique())

In [ ]:
# ============================================================
# RECOMPUTE lab_day AND RESTRICT TO DAYS 0–7
# ============================================================

df_labs_cohort["lab_day"] = (
    df_labs_cohort["Date"].dt.normalize()
    - df_labs_cohort["startDate"].dt.normalize()
).dt.days

df_labs_cohort = df_labs_cohort[
    df_labs_cohort["lab_day"].between(0, 7)
].copy()

df_labs_cohort = df_labs_cohort[
    df_labs_cohort["lab_name"].isin(CORE_BIOMARKERS)
].copy()

print("Labs in final cohort, days 0–7:", df_labs_cohort.shape)
print("Episodes with labs days 0–7:", df_labs_cohort["episode_key"].nunique())
print("Biomarkers:")
print(sorted(df_labs_cohort["lab_name"].unique()))

In [ ]:
# ============================================================
# DAILY AGGREGATION
# ============================================================

df_daily = (
    df_labs_cohort
    .groupby(
        ["episode_key", "lab_name", "lab_day"],
        as_index=False
    )["Numeric_value"]
    .median()
    .rename(columns={
        "lab_name": "biomarker",
        "Numeric_value": "value"
    })
)

print("Daily aggregated labs:", df_daily.shape)
print("Episodes:", df_daily["episode_key"].nunique())
display(df_daily.head())

In [ ]:
# ============================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_window_features(df_daily, window_name, start_day, end_day):
    """
    Extract laboratory features for a given time window.

    mean_window:
        Mean of all observed daily values within the window.

    delta_window:
        Last observed daily value - first observed daily value.
        Computed only if >=2 distinct measurement days are available.

    n_tests_window:
        Number of observed daily measurements within the window.

    has_dynamic_window:
        1 if delta is observable, i.e. >=2 distinct measurement days.
        0 otherwise.
    """

    w = df_daily[
        df_daily["lab_day"].between(start_day, end_day)
    ].copy()

    if w.empty:
        return pd.DataFrame({"episode_key": []})

    # Mean
    mean_df = (
        w.groupby(["episode_key", "biomarker"])["value"]
        .mean()
        .unstack("biomarker")
    )

    mean_df.columns = [
        f"{biomarker}_mean_{window_name}"
        for biomarker in mean_df.columns
    ]

    # Number of observed laboratory days in the window
    # # Note: after daily median aggregation, this is not the raw number of tests,
    # # but the number of distinct daily biomarker observations.
    n_days_df = (
       w.groupby(["episode_key", "biomarker"])["value"]
       .count()
       .unstack("biomarker")
    )

    n_days_df.columns = [
       f"{biomarker}_n_days_{window_name}"
       for biomarker in n_days_df.columns
    ]

    # Delta and dynamic flag
    w_sorted = w.sort_values(
        ["episode_key", "biomarker", "lab_day"]
    )

    first_values = (
        w_sorted
        .groupby(["episode_key", "biomarker"], as_index=False)
        .first()
        [["episode_key", "biomarker", "lab_day", "value"]]
        .rename(columns={
            "lab_day": "first_day",
            "value": "first_value"
        })
    )

    last_values = (
        w_sorted
        .groupby(["episode_key", "biomarker"], as_index=False)
        .last()
        [["episode_key", "biomarker", "lab_day", "value"]]
        .rename(columns={
            "lab_day": "last_day",
            "value": "last_value"
        })
    )

    n_days = (
        w_sorted
        .groupby(["episode_key", "biomarker"])["lab_day"]
        .nunique()
        .reset_index(name="n_distinct_days")
    )

    delta_long = (
        first_values
        .merge(last_values, on=["episode_key", "biomarker"], how="left")
        .merge(n_days, on=["episode_key", "biomarker"], how="left")
    )

    delta_long[f"has_dynamic_{window_name}"] = np.where(
        delta_long["n_distinct_days"] >= 2,
        1,
        0
    )

    delta_long[f"delta_{window_name}"] = np.where(
        delta_long[f"has_dynamic_{window_name}"] == 1,
        delta_long["last_value"] - delta_long["first_value"],
        np.nan
    )

    delta_df = delta_long.pivot(
        index="episode_key",
        columns="biomarker",
        values=f"delta_{window_name}"
    )

    delta_df.columns = [
        f"{biomarker}_delta_{window_name}"
        for biomarker in delta_df.columns
    ]

    dynamic_df = delta_long.pivot(
        index="episode_key",
        columns="biomarker",
        values=f"has_dynamic_{window_name}"
    )

    dynamic_df.columns = [
        f"{biomarker}_has_dynamic_{window_name}"
        for biomarker in dynamic_df.columns
    ]

    features = pd.concat(
        [mean_df, n_days_df, delta_df, dynamic_df],
        axis=1
    ).reset_index()

    return features

In [ ]:
# ============================================================
# EXTRACT LABORATORY FEATURES
# ============================================================

feature_tables = []

# Full feature extraction for 0_3 and 3_7
for window_name in ["0_3", "3_7"]:
    start_day, end_day = LAB_WINDOWS[window_name]

    ft = extract_window_features(
        df_daily=df_daily,
        window_name=window_name,
        start_day=start_day,
        end_day=end_day
    )

    feature_tables.append(ft)

# For 0_7 keep only delta and has_dynamic
start_day, end_day = LAB_WINDOWS["0_7"]

ft_0_7 = extract_window_features(
    df_daily=df_daily,
    window_name="0_7",
    start_day=start_day,
    end_day=end_day
)

cols_0_7 = (
    ["episode_key"]
    + [c for c in ft_0_7.columns if "_delta_0_7" in c]
    + [c for c in ft_0_7.columns if "_has_dynamic_0_7" in c]
)

ft_0_7 = ft_0_7[cols_0_7].copy()

feature_tables.append(ft_0_7)

for i, ft in enumerate(feature_tables):
    print(f"Feature table {i}:", ft.shape)

In [ ]:
# ============================================================
# DELTA COLLINEARITY NOTE
# ============================================================

print("""
Delta features are intentionally retained for interpretability:
- delta_0_3 captures early within-window change.
- delta_3_7 captures later within-window change.
- delta_0_7 captures global first-week change.

These variables are partially correlated and should not be interpreted as
independent biological effects. Regularized models downstream help reduce
instability due to collinearity.
""")

In [ ]:
# ============================================================
# MERGE FEATURE TABLES
# ============================================================

df_lab_features = feature_tables[0].copy()

for ft in feature_tables[1:]:
    df_lab_features = df_lab_features.merge(
        ft,
        on="episode_key",
        how="outer"
    )

print("Lab feature table:", df_lab_features.shape)
print("Episodes with lab features:", df_lab_features["episode_key"].nunique())

display(df_lab_features.head())

In [ ]:
# ============================================================
# FILL COUNT AND DYNAMIC INDICATORS
# ============================================================

count_cols = [c for c in df_lab_features.columns if "_n_days_" in c]
dynamic_cols = [c for c in df_lab_features.columns if "_has_dynamic_" in c]

df_lab_features[count_cols] = df_lab_features[count_cols].fillna(0)
df_lab_features[dynamic_cols] = df_lab_features[dynamic_cols].fillna(0)

for col in count_cols + dynamic_cols:
    df_lab_features[col] = df_lab_features[col].astype(int)

print("Daily observation count features:", len(count_cols))
print("Dynamic indicators:", len(dynamic_cols))

In [ ]:
# ============================================================
# GLOBAL LAB AVAILABILITY
# ============================================================

global_lab = (
    df_daily
    .groupby("episode_key")
    .agg(
        n_total_lab_records_0_7=("value", "count"),
        n_total_lab_days_0_7=("lab_day", "nunique"),
        n_biomarkers_measured_0_7=("biomarker", "nunique")
    )
    .reset_index()
)

global_lab["has_any_lab_0_7"] = 1
global_lab["labs_per_day_0_7"] = (
    global_lab["n_total_lab_records_0_7"]
    / global_lab["n_total_lab_days_0_7"].replace(0, np.nan)
)

df_lab_features = df_lab_features.merge(
    global_lab,
    on="episode_key",
    how="left"
)

for col in [
    "n_total_lab_records_0_7",
    "n_total_lab_days_0_7",
    "n_biomarkers_measured_0_7",
    "has_any_lab_0_7",
]:
    df_lab_features[col] = df_lab_features[col].fillna(0).astype(int)

df_lab_features["labs_per_day_0_7"] = (
    df_lab_features["labs_per_day_0_7"]
    .fillna(0)
    .astype(float)
)

# Early laboratory monitoring intensity: days 0–1
early_labs = (
    df_daily[df_daily["lab_day"].between(0, 1)]
    .groupby("episode_key")
    .size()
    .reset_index(name="n_labs_day0_1")
)

df_lab_features = df_lab_features.merge(
    early_labs,
    on="episode_key",
    how="left"
)

df_lab_features["n_labs_day0_1"] = (
    df_lab_features["n_labs_day0_1"]
    .fillna(0)
    .astype(int)
)

print(df_lab_features["has_any_lab_0_7"].value_counts())

In [ ]:
# NOTE:
# n_total_lab_records_0_7 is computed after daily aggregation.
# Therefore, despite the name, it represents the number of daily biomarker-level
# observations within days 0–7, not the number of raw laboratory records.
# It is retained with this name for downstream compatibility.

In [ ]:
# ============================================================
# MERGE FEATURES WITH FINAL COHORT
# ============================================================

df_model = df_cohort.merge(
    df_lab_features,
    on="episode_key",
    how="left"
)

# Fill lab availability for patients without lab features
for col in [
    "n_total_lab_records_0_7",
    "n_total_lab_days_0_7",
    "n_biomarkers_measured_0_7",
    "has_any_lab_0_7",
    "labs_per_day_0_7",
    "n_labs_day0_1",
]:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna(0)

for col in [
    "n_total_lab_records_0_7",
    "n_total_lab_days_0_7",
    "n_biomarkers_measured_0_7",
    "has_any_lab_0_7",
    "n_labs_day0_1",
]:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(int)

if "labs_per_day_0_7" in df_model.columns:
    df_model["labs_per_day_0_7"] = df_model["labs_per_day_0_7"].astype(float)

print("Final model dataset:", df_model.shape)
print("Patients:", df_model["pubID"].nunique())
print("Episodes:", df_model["episode_key"].nunique())
print("Patients with any labs 0–7:")
print(df_model["has_any_lab_0_7"].value_counts(dropna=False))

In [ ]:
# ============================================================
# MISSINGNESS SUMMARY
# ============================================================

lab_feature_cols = [
    c for c in df_model.columns
    if (
        "_mean_" in c
        or "_delta_" in c
        or "_n_days_" in c
        or "_has_dynamic_" in c
        or c in [
            "n_total_lab_records_0_7",
            "n_total_lab_days_0_7",
            "n_biomarkers_measured_0_7",
            "has_any_lab_0_7",
            "labs_per_day_0_7",
            "n_labs_day0_1",
        ]
    )
]

missingness_rows = []

for col in lab_feature_cols:
    missingness_rows.append({
        "feature": col,
        "missing_n": df_model[col].isna().sum(),
        "missing_pct": df_model[col].isna().mean() * 100,
        "available_n": df_model[col].notna().sum(),
    })

missingness_labs = (
    pd.DataFrame(missingness_rows)
    .sort_values("missing_pct", ascending=False)
)

display(missingness_labs.head(50))

In [ ]:
# ============================================================
# SANITY CHECKS
# ============================================================

expected_patterns = [
    "_mean_0_3",
    "_mean_3_7",
    "_delta_0_3",
    "_delta_3_7",
    "_delta_0_7",
    "_n_days_0_3",
    "_n_days_3_7",
    "_has_dynamic_0_3",
    "_has_dynamic_3_7",
    "_has_dynamic_0_7",
]

print("Expected feature counts:")
for pattern in expected_patterns:
    cols = [c for c in df_model.columns if pattern in c]
    print(pattern, len(cols))

old_patterns = ["_4_7", "_0_5", "_3_5"]
old_cols = [
    c for c in df_model.columns
    if any(p in c for p in old_patterns)
]

print("\nOld-window columns:", old_cols)
assert len(old_cols) == 0

mean_cols = [c for c in df_model.columns if "_mean_" in c]
delta_cols = [c for c in df_model.columns if "_delta_" in c]
count_cols = [c for c in df_model.columns if "_n_days_" in c]
dynamic_cols = [c for c in df_model.columns if "_has_dynamic_" in c]

print("\nFeature groups:")
print("Mean:", len(mean_cols))
print("Delta:", len(delta_cols))
print("Daily observation count:", len(count_cols))
print("Dynamic:", len(dynamic_cols))

In [ ]:
# ============================================================
# TRAJECTORY OBSERVABILITY SUMMARY
# ============================================================

trajectory_summary = []

for biomarker in CORE_BIOMARKERS:

    row = {
        "biomarker": biomarker,

        "mean_0_3_available_pct":
            df_model[f"{biomarker}_mean_0_3"].notna().mean() * 100,

        "mean_3_7_available_pct":
            df_model[f"{biomarker}_mean_3_7"].notna().mean() * 100,

        "delta_0_3_available_pct":
            df_model[f"{biomarker}_delta_0_3"].notna().mean() * 100,

        "delta_3_7_available_pct":
            df_model[f"{biomarker}_delta_3_7"].notna().mean() * 100,

        "delta_0_7_available_pct":
            df_model[f"{biomarker}_delta_0_7"].notna().mean() * 100,
    }

    trajectory_summary.append(row)

trajectory_summary = pd.DataFrame(trajectory_summary)

display(
    trajectory_summary.sort_values(
        "delta_3_7_available_pct",
        ascending=False
    )
)

In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

df_model.to_parquet(
    OUTPUT_03 / "03_model_features.parquet",
    index=False
)

missingness_labs.to_excel(
    OUTPUT_03 / "03_missingness_labs.xlsx",
    index=False
)

df_lab_features.to_parquet(
    OUTPUT_03 / "03_lab_features_only.parquet",
    index=False
)

df_daily.to_parquet(
    OUTPUT_03 / "03_daily_labs.parquet",
    index=False
)

trajectory_summary.to_excel(
    OUTPUT_03 / "03_trajectory_observability_summary.xlsx",
    index=False
)

print("Saved:")
print(OUTPUT_03 / "03_model_features.parquet")
print(OUTPUT_03 / "03_missingness_labs.xlsx")
print(OUTPUT_03 / "03_lab_features_only.parquet")
print(OUTPUT_03 / "03_daily_labs.parquet")

In [ ]:
# ============================================================
# OUTPUT EXISTENCE QC
# ============================================================

expected_outputs = [
    OUTPUT_03 / "03_model_features.parquet",
    OUTPUT_03 / "03_lab_features_only.parquet",
    OUTPUT_03 / "03_daily_labs.parquet",
    OUTPUT_03 / "03_missingness_labs.xlsx",
    OUTPUT_03 / "03_trajectory_observability_summary.xlsx",
]

for path in expected_outputs:
    print(path.name, "exists:", path.exists())
    assert path.exists(), f"Missing expected output: {path}"

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 80)
print("03 FEATURE ENGINEERING SUMMARY")
print("=" * 80)

print("\nInput cohort")
print("Rows:", df_cohort.shape[0])
print("Patients:", df_cohort["pubID"].nunique())
print("Episodes:", df_cohort["episode_key"].nunique())

print("\nLaboratory data")
print("Raw selected lab rows:", df_labs.shape[0])
print("Labs assigned to final cohort days 0–7:", df_labs_cohort.shape[0])
print("Episodes with labs days 0–7:", df_labs_cohort["episode_key"].nunique())

print("\nFinal model dataset")
print("Rows:", df_model.shape[0])
print("Columns:", df_model.shape[1])
print("Patients:", df_model["pubID"].nunique())
print("Episodes:", df_model["episode_key"].nunique())

print("\nFeature groups")
print("Mean features:", len(mean_cols))
print("Delta features:", len(delta_cols))
print("Daily observation count features:", len(count_cols))
print("Global lab intensity features:", 6)
print("Dynamic indicators:", len(dynamic_cols))

print("\nLab availability")
print(df_model["has_any_lab_0_7"].value_counts(dropna=False))